# 3.4 — Use Snowflake AI Observability Tools

**Exam domain:** Gen AI Governance · **Weight:** 28%

## The problem this solves

The support chatbot went live and the complaints are vague: "the answers are wrong", "it makes things
up", "it did not answer what I asked". You cannot fix any of that, because all three describe the same
symptom from the user's side and have completely different causes — bad retrieval, a model ignoring
good context, or a model answering a question nobody asked.

AI Observability turns those complaints into numbers you can act on. It records what the application
actually did on each request, then scores those records against named metrics so you can tell the
three failures apart.

## What you will be able to do

- Name the five evaluation metrics and say which question each answers
- Read context relevance and groundedness together to separate a retrieval failure from a generation failure
- Instrument a RAG application with TruLens so its spans are captured
- Run an evaluation over a dataset and compute metrics on it
- Find traces and scores in SQL, and know which view is a billing view rather than a quality one

## Before you start

- Notebooks 3.1 and 3.2, for the Cortex privileges everything here depends on
- A Python environment with `trulens-core` and `trulens-connectors-snowflake` installed
- A Cortex Search service and a table to answer questions over — the repo setup script creates both

📖 **Snowflake documentation for this notebook**
- [AI Observability with Snowflake Cortex](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability)
- [AI Observability reference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability/reference)
- [Evaluate AI applications](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability)
- [Trace applications with TruLens](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability/trace-applications-trulens)
- [Cortex Agent evaluations](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-evaluations)
- [CORTEX_ANALYST_USAGE_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_analyst_usage_history)


---
## What AI Observability records, and where it goes

Three things, and it helps to keep them apart:

- **Tracing** captures the span tree of a single request — the user's input, the retrieval step, the
  generation step, the final answer. A **span** is one instrumented operation, and spans nest.
- **Evaluation** runs your application over a dataset and scores each record with LLM-as-a-judge
  metrics.
- **Comparison** puts two runs side by side — two prompts, two models, two versions of the app.

### Where the data lives

```
SNOWFLAKE.LOCAL.AI_OBSERVABILITY_EVENTS
```

Traces and evaluation results for external agents, Cortex Agents and CoWork land here. Cortex Analyst
requests made directly have their own location, `SNOWFLAKE.LOCAL.CORTEX_ANALYST_REQUESTS_RAW`.

An **event table** is Snowflake's OpenTelemetry-shaped log sink. Its columns are `TIMESTAMP`,
`START_TIMESTAMP`, `OBSERVED_TIMESTAMP`, `TRACE`, `RESOURCE`, `RESOURCE_ATTRIBUTES`, `SCOPE`,
`SCOPE_ATTRIBUTES`, `RECORD_TYPE`, `RECORD`, `RECORD_ATTRIBUTES`, `VALUE` and `EXEMPLARS`, and
`RECORD_TYPE` takes the values `LOG`, `SPAN`, `SPAN_EVENT`, `METRIC` and `EVENT`. Traces are `SPAN`
rows; the interesting detail sits in the semi-structured attribute columns.

In Snowsight, TruLens-instrumented external agents show under **AI & ML » Evaluations**; Cortex Agents
and CoWork have their own **Observability** and **Evaluations** tabs.

### Privileges

- The `SNOWFLAKE.CORTEX_USER` database role
- `CREATE EXTERNAL AGENT` on the schema, to register an application
- `CREATE TASK` on the schema and the global `EXECUTE TASK` privilege — evaluation runs execute as tasks
- `USE AI FUNCTIONS` on the account, because the judges are themselves AI functions

That last one surprises people: an evaluation costs credits, because scoring a run means calling a
model once per record per metric.

→ [More on AI Observability](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability)
→ [More on event table columns](https://docs.snowflake.com/en/developer-guide/logging-tracing/event-table-columns)


---
## The five metrics, and the attributes each one needs

Each metric is computed from specific span attributes. If the attribute is not captured, the metric
cannot be computed — which is why instrumentation and evaluation are the same topic.

| Metric | Question it answers | Ground truth? | Attributes required |
|---|---|---|---|
| **Context Relevance** | Is the retrieved context relevant to the user's query? | No | `RETRIEVAL.QUERY_TEXT`, `RETRIEVAL.RETRIEVED_CONTEXTS` |
| **Groundedness** | Is the response supported by and grounded in the retrieved context? | No | `RETRIEVAL.RETRIEVED_CONTEXTS`, `RECORD_ROOT.OUTPUT` |
| **Answer Relevance** | Is the response relevant to the user's query? | No | `RECORD_ROOT.INPUT`, `RECORD_ROOT.OUTPUT` |
| **Correctness** | How aligned is the response with the ground truth? | **Yes** | `RECORD_ROOT.INPUT`, `RECORD_ROOT.GROUND_TRUTH_OUTPUT`, `RECORD_ROOT.OUTPUT` |
| **Coherence** | Is the response coherent, without logical gaps or contradictions? | No | `RECORD_ROOT.OUTPUT` |

**Cost** and **latency** are captured per run alongside them.

The first three are the classic RAG triad. Read the first two together and the fault localises itself:

```
Context Relevance LOW   -> retrieval problem   (chunking, embedding model, filters, reranking)
Groundedness      LOW   -> generation problem  (hallucination: constrain the prompt, temperature 0)
Answer Relevance  LOW   -> the model answered a different question (prompt/instruction problem)
Correctness       LOW   -> right shape, wrong facts — and undetectable without ground truth
```

### Agents are scored differently

Cortex Agent evaluations use their own system metrics, because an agent can fail at planning rather
than at wording: **answer correctness**, **tool selection accuracy** (did the orchestrator call the
tools you expected?), **tool execution accuracy** (did each tool get sensible input and return usable
output?) and **logical consistency**, which is reference-free. You can add custom LLM-judged metrics on
top. Results are read in the Evaluations tab or with `GET_AI_EVALUATION_DATA`, `GET_AI_RECORD_TRACE`
and `GET_AI_OBSERVABILITY_LOGS`, and come back with columns including `RECORD_ID`, `INPUT`, `OUTPUT`,
`EVAL_AGG_SCORE` and `METRIC_NAME`.

→ [More on the metrics and their required attributes](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability/reference)
→ [More on agent evaluations](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-evaluations)


In [ ]:
# ============================================================
# TruLens on Snowflake
# Install (once):  pip install trulens-core trulens-connectors-snowflake
# ============================================================
from trulens.core.otel.instrument import instrument
from trulens.otel.semconv.trace import SpanAttributes
from trulens.apps.app import TruApp
from trulens.connectors.snowflake import SnowflakeConnector
from trulens.core.run import Run, RunConfig

from snowflake.snowpark.context import get_active_session

session = get_active_session()

# The connector registers the app and writes spans into SNOWFLAKE.LOCAL.AI_OBSERVABILITY_EVENTS
connector = SnowflakeConnector(snowpark_session=session)

print("TruLens connector ready — traces land in SNOWFLAKE.LOCAL.AI_OBSERVABILITY_EVENTS")
print("View runs in Snowsight -> AI & ML -> Evaluations")


---
## Instrumenting an application

The `@instrument` decorator marks a method as a span and maps its arguments and return value onto
named attributes. The span **type** is what tells the evaluator which value is the query, which is the
retrieved context, and which is the final answer — get the types wrong and the metrics compute against
the wrong strings rather than failing loudly.

Three span types matter here: `RECORD_ROOT` (the application's entry point), `RETRIEVAL` (search) and
`GENERATION` (the model call).

→ [More on tracing with TruLens](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability/trace-applications-trulens)


In [ ]:
# ============================================================
# Instrument a RAG app. Span types tell TruLens which value is the query,
# which is the retrieved context, and which is the final answer.
# ============================================================
from trulens.core.otel.instrument import instrument
from trulens.otel.semconv.trace import SpanAttributes


class SupportRAG:

    @instrument(
        span_type=SpanAttributes.SpanType.RETRIEVAL,
        attributes={
            SpanAttributes.RETRIEVAL.QUERY_TEXT:       "query",
            SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS: "return",
        },
    )
    def retrieve(self, query: str):
        """Retrieve grounding context from a Cortex Search service."""
        rows = session.sql(
            """SELECT PARSE_JSON(
                     SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
                         'GENAI_STUDY.PUBLIC.TICKET_SEARCH',
                         OBJECT_CONSTRUCT('query', ?, 'columns', ARRAY_CONSTRUCT('safe_text'),
                                          'limit', 3)::VARCHAR
                     )
                   )['results'] AS r""",
            params=[query],
        ).collect()
        return [row["R"] for row in rows]

    @instrument(span_type=SpanAttributes.SpanType.GENERATION)
    def generate(self, query: str, context: list) -> str:
        return session.sql(
            """SELECT AI_COMPLETE('llama3.1-8b',
                     'Answer ONLY from the context. If it is not there, say you do not know.\n'
                     || 'Context: ' || ? || '\nQuestion: ' || ?) AS a""",
            params=[" ".join(map(str, context)), query],
        ).collect()[0]["A"]

    @instrument(span_type=SpanAttributes.SpanType.RECORD_ROOT,
                attributes={SpanAttributes.RECORD_ROOT.INPUT:  "query",
                            SpanAttributes.RECORD_ROOT.OUTPUT: "return"})
    def answer_query(self, query: str) -> str:
        return self.generate(query, self.retrieve(query))


rag = SupportRAG()

tru_app = TruApp(
    rag,
    app_name="support_rag",
    app_version="v1",
    connector=connector,
    main_method=rag.answer_query,
)

print("App registered. RECORD_ROOT / RETRIEVAL / GENERATION spans will be captured per call.")


---
## Running an evaluation

A run has three moving parts:

1. A **dataset** — a dataframe or a table, with a column per input and, if you want correctness, a
   column of known-good answers.
2. A **`dataset_spec`** mapping span attributes to dataset columns, for example `RECORD_ROOT.INPUT` to
   your query column and `RECORD_ROOT.GROUND_TRUTH_OUTPUT` to your expected-answer column.
3. An **LLM judge**, named in `RunConfig`, that scores the records.

`run.start(...)` executes the application over every row and captures traces. `run.compute_metrics([…])`
scores them afterwards. The two are separate steps because you can score the same run against more
metrics later without re-running the application.

**The trade-off in choosing a judge.** A large judge model gives scores you trust more and costs more
per record; a small one lets you evaluate nightly over a big dataset for less. Since the judges are AI
functions billed like any other, your evaluation budget competes directly with your production budget.

→ [More on evaluating AI applications](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability)


In [ ]:
# ============================================================
# Configure and execute an evaluation run, then compute metrics.
# ============================================================
import pandas as pd
from trulens.core.run import RunConfig

eval_df = pd.DataFrame({
    "query": [
        "What are the most critical technical issues?",
        "Which customers have billing complaints?",
        "Are there any security-related tickets?",
    ],
    "expected_answer": [
        "Application crashes on version 3.x after the latest update.",
        "Customers reporting duplicate charges on their monthly invoice.",
        "Tickets describing unauthorized account access.",
    ],
})

run_config = RunConfig(
    run_name="support_rag_eval_1",
    dataset_name="support_eval_set",
    source_type="DATAFRAME",
    dataset_spec={
        "RECORD_ROOT.INPUT":                "query",
        "RECORD_ROOT.GROUND_TRUTH_OUTPUT":  "expected_answer",
    },
    llm_judge_name="mistral-large3",     # the judge model for LLM-as-a-judge metrics
)

run = tru_app.add_run(run_config=run_config)
run.start(input_df=eval_df)              # executes the app over every row, capturing traces

# Compute the documented metrics. Correctness needs GROUND_TRUTH_OUTPUT in dataset_spec.
run.compute_metrics(metrics=[
    "context_relevance",
    "groundedness",
    "answer_relevance",
    "correctness",
    "coherence",
])

print(run.get_status())
print("Results: Snowsight -> AI & ML -> Evaluations, or query SNOWFLAKE.LOCAL.AI_OBSERVABILITY_EVENTS")


> ### ⚠️ Common misconceptions
>
> **"`CORTEX_ANALYST_USAGE_HISTORY` will show me the questions users asked and the SQL Analyst
> generated."**
> It will not. Its columns are `START_TIME`, `END_TIME`, `REQUEST_COUNT`, `CREDITS` and `USERNAME` —
> it is a billing view. Question and SQL content live in the Analyst request logging under
> `SNOWFLAKE.LOCAL.CORTEX_ANALYST_REQUESTS_RAW`. Querying the billing view for content returns a column
> that does not exist, not an empty result.
> → [CORTEX_ANALYST_USAGE_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_analyst_usage_history)
>
> **"Groundedness tells me whether the answer is right."**
> It tells you whether the answer is supported by the retrieved context. An answer perfectly grounded
> in a wrong document scores high and is still wrong. Only **correctness** compares against a known
> right answer, and it needs `RECORD_ROOT.GROUND_TRUTH_OUTPUT` in the dataset spec to compute at all.
> → [AI Observability reference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability/reference)
>
> **"I enabled tracing, so I will get metric scores."**
> Tracing captures spans; metrics are computed by a separate `compute_metrics` step, and each metric
> needs its specific attributes present in those spans. A retrieval span without
> `RETRIEVAL.RETRIEVED_CONTEXTS` means context relevance and groundedness cannot be computed — you get
> no score rather than a bad score.
> → [Evaluate AI applications](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability)
>
> **"Evaluation is free — it is just reading logs."**
> The judges are AI functions and the runs execute as tasks. That is why the privilege list includes
> `USE AI FUNCTIONS`, `CREATE TASK` and `EXECUTE TASK`, and why a nightly evaluation over a large
> dataset shows up on the Cortex bill.
> → [AI Observability reference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability/reference)


In [ ]:
%%sql -r observability_events
-- ============================================================
-- Traces and evaluation scores, queried directly.
-- The event table follows the OpenTelemetry span shape; RECORD_TYPE is one of
-- LOG, SPAN, SPAN_EVENT, METRIC or EVENT, and traces are SPAN rows.
-- ============================================================
SELECT
    TIMESTAMP,
    RECORD_TYPE,
    RESOURCE_ATTRIBUTES,
    RECORD_ATTRIBUTES,
    RECORD
FROM SNOWFLAKE.LOCAL.AI_OBSERVABILITY_EVENTS
WHERE TIMESTAMP >= DATEADD('day', -7, CURRENT_TIMESTAMP())
ORDER BY TIMESTAMP DESC
LIMIT 20;

-- Inspect one row and confirm the attribute keys before hard-coding paths into a dashboard —
-- span attribute names follow the TruLens semantic conventions and are versioned with them.
-- GET_AI_OBSERVABILITY_EVENTS(<db>, <schema>, <object name>, 'EXTERNAL AGENT') is the supported
-- way to read the events for one registered application.

-- Account event table (a separate feature) for general application telemetry:
-- ALTER ACCOUNT SET EVENT_TABLE = <db>.<schema>.<event_table>;


In [ ]:
%%sql -r analyst_observability
-- ============================================================
-- Cortex Analyst monitoring
-- CORTEX_ANALYST_USAGE_HISTORY is a BILLING view. Its five columns are
--   START_TIME, END_TIME, REQUEST_COUNT, CREDITS, USERNAME
-- It holds no question text, no generated SQL and no feedback.
-- ============================================================
SELECT
    DATE_TRUNC('day', START_TIME) AS day,
    USERNAME,
    SUM(REQUEST_COUNT)            AS messages,
    SUM(CREDITS)                  AS credits
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_ANALYST_USAGE_HISTORY
WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
GROUP BY 1, 2
ORDER BY 1 DESC, credits DESC;

-- For question, generated-SQL and feedback content, use the Analyst request logging in
-- SNOWFLAKE.LOCAL.CORTEX_ANALYST_REQUESTS_RAW.

-- Spend and quality live in different places:
--   cost    -> SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY
--   quality -> SNOWFLAKE.LOCAL.AI_OBSERVABILITY_EVENTS


> ### 🤔 Stop and think
>
> - A nightly evaluation gives you a trend line and costs credits every night. Evaluating only before a
>   release costs less and tells you nothing about drift. Which failure would hurt your users more, and
>   how would you know it was happening?
> - Correctness needs ground-truth answers, which someone has to write and keep current as the data
>   changes. Who owns that dataset in your team, and what happens to your quality signal the month they
>   are busy?
> - Cost and quality metrics live in different places — credits in `ACCOUNT_USAGE`, scores in the
>   observability event table. If a model swap cut spend 40% and groundedness 5 points, who in your
>   organisation decides whether that was a good trade, and on what evidence?


---
## Worked scenario: "the answers are made up"

**The situation.** After deploying a Cortex Search plus Cortex Analyst chatbot, users complain that
answers are invented and do not match the data. Which metric identifies the root cause, and what is
the fix?

**The metric is groundedness.** A low groundedness score means the generated response is not supported
by the retrieved context — which is what hallucination means in a RAG system.

**Read it with context relevance, and the fault localises:**

| Context Relevance | Groundedness | Diagnosis |
|---|---|---|
| Low | Low | **Retrieval** is failing — the model had nothing good to work with |
| High | Low | **Generation** is failing — good context, the model ignored it |
| High | High, users still unhappy | Check **answer relevance** (wrong question answered), then **correctness** (needs ground truth) |

**Fixes, in order of effort:**

1. Constrain the prompt: *"Answer only from the provided context. If it is not there, say you do not know."*
2. `temperature: 0` — already the default for `AI_COMPLETE` — plus a `response_format` schema so the
   shape of the answer is fixed.
3. Improve retrieval: smaller chunks, a stronger embedding model, filters on indexed attribute columns.
4. Cortex Guard, `model_parameters => {'guardrails': TRUE}`, for output safety — which is a different
   problem and will not help here.
5. Fine-tune, if the failures are consistently about domain vocabulary rather than missing context.

**Then operationalise it.** Register the app with `TruApp`, run an evaluation over a fixed dataset on a
schedule, and alert when average groundedness falls below your threshold. Keep ground-truth answers in
that dataset so correctness is available too — without it, an answer that is grounded in the wrong
document looks healthy.


---

## Check your understanding

Twelve questions on this notebook. Answer before expanding.

**1.** Where do AI Observability traces and evaluation results land?

<details><summary>Show answer</summary>

`SNOWFLAKE.LOCAL.AI_OBSERVABILITY_EVENTS`, an event table in the OpenTelemetry shape. Cortex Analyst
requests made directly are the exception — those go to `SNOWFLAKE.LOCAL.CORTEX_ANALYST_REQUESTS_RAW`.
Nothing quality-related lands in `ACCOUNT_USAGE`; that schema is for credits.

→ [AI Observability](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability)

</details>

**2.** Which of the five metrics requires ground truth?

<details><summary>Show answer</summary>

Correctness. It measures how aligned the response is with the ground truth, so it needs
`RECORD_ROOT.GROUND_TRUTH_OUTPUT` in the dataset spec. Coherence is the tempting distractor — it is not
part of the RAG triad either, but it scores the response on its own and needs no reference.

→ [AI Observability reference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability/reference)

</details>

**3.** Which privileges does running an evaluation require beyond `SNOWFLAKE.CORTEX_USER`?

<details><summary>Show answer</summary>

`CREATE EXTERNAL AGENT` on the schema to register the application, `CREATE TASK` on the schema and the
global `EXECUTE TASK` privilege because runs execute as tasks, and `USE AI FUNCTIONS` on the account
because the judges are AI functions.

→ [AI Observability reference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability/reference)

</details>

**4.** A run reports answer relevance and coherence but no groundedness or context relevance. What is
wrong?

<details><summary>Show answer</summary>

The retrieval span is not capturing the attributes those two metrics need —
`RETRIEVAL.QUERY_TEXT` and `RETRIEVAL.RETRIEVED_CONTEXTS`. The two metrics that only need
`RECORD_ROOT.INPUT` and `RECORD_ROOT.OUTPUT` computed fine, which is why the run looks partly
successful rather than broken.

→ [AI Observability reference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability/reference)

</details>

**5.** Context relevance is high, groundedness is low. Retrieval problem or generation problem?

<details><summary>Show answer</summary>

Generation. The retriever found relevant context and the model produced an answer that is not
supported by it. Improving retrieval here spends effort on the half that already works — constrain the
prompt with a refusal instruction and pin `temperature` to 0 instead.

→ [AI Observability reference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability/reference)

</details>

**6.** You query `CORTEX_ANALYST_USAGE_HISTORY` for the generated SQL and get an error about an unknown
column. Where should you be looking?

<details><summary>Show answer</summary>

`SNOWFLAKE.LOCAL.CORTEX_ANALYST_REQUESTS_RAW`. The `ACCOUNT_USAGE` view is a billing view with exactly
five columns — `START_TIME`, `END_TIME`, `REQUEST_COUNT`, `CREDITS`, `USERNAME` — and carries no
request content at all.

→ [CORTEX_ANALYST_USAGE_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_analyst_usage_history)

</details>

**7.** Groundedness averages 0.9 and users still say the answers are wrong. What do you check next?

<details><summary>Show answer</summary>

Answer relevance first — a well-grounded answer to a question the user did not ask — and then
correctness, which needs ground-truth answers in the dataset. A high groundedness score with wrong
answers usually means the retriever is confidently returning the wrong document: the answer faithfully
reflects it, and no reference-free metric can notice.

→ [AI Observability reference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability/reference)

</details>

**8.** An agent picks the search tool when it should have used the text-to-SQL tool. Which evaluation
metric catches this?

<details><summary>Show answer</summary>

Tool selection accuracy, one of the Cortex Agent evaluation system metrics — it measures whether the
orchestration layer invokes the tools you expect for the user's goal. The RAG triad would not catch it:
the agent retrieved something relevant and answered coherently from it, just via the wrong route.

→ [Cortex Agent evaluations](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-evaluations)

</details>

**9.** What is the difference between `run.start(...)` and `run.compute_metrics([...])`?

<details><summary>Show answer</summary>

`run.start(...)` executes the application over every row of the dataset and captures the traces.
`run.compute_metrics([...])` scores those captured records with the named metrics. They are separate so
you can add metrics to an existing run later without re-running the application — and so the cost of
scoring is separable from the cost of running.

→ [Evaluate AI applications](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability)

</details>

**10.** You are choosing between running evaluations nightly and running them at each release. What
does each choice buy and cost?

<details><summary>Show answer</summary>

Nightly gives you a drift signal — model updates, data shifts and retrieval degradation show up within
a day — and bills judge-model credits every night, on top of the app's own run. Per-release is far
cheaper and blind between releases, which matters because none of those drift sources wait for your
release calendar. A middle path is a small nightly dataset plus a full suite at release.

→ [Evaluate AI applications](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability)

</details>

**11.** A colleague proposes turning on Cortex Guard to fix low groundedness scores. What do you tell
them?

<details><summary>Show answer</summary>

Guard filters potentially unsafe and harmful responses. It has no view on whether an answer is
supported by the retrieved context, so groundedness will not move — and you will pay for the filtering.
The fix is grounding: a refusal instruction, `temperature: 0`, and better retrieval.

→ [AI_COMPLETE (single string)](https://docs.snowflake.com/en/sql-reference/functions/ai_complete-single-string)

</details>

**12.** Connecting to cost governance: you want one dashboard showing both quality and spend for the
chatbot. Which two sources do you join, and on what?

<details><summary>Show answer</summary>

Quality comes from `SNOWFLAKE.LOCAL.AI_OBSERVABILITY_EVENTS`; spend comes from
`SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY`, or
`CORTEX_AGENT_USAGE_HISTORY` if the chatbot is an agent. There is no shared key handed to you — the
practical join is time plus a `QUERY_TAG` you set deliberately on the production path, which is why
tagging is worth doing before you need the dashboard. Notebook 3.3 covers the cost side.

→ [CORTEX_AI_FUNCTIONS_USAGE_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_ai_functions_usage_history)

</details>
